# 11 - Final Tables and Figures

**Purpose:** Produce final tables and figures with consistent styling.

**Inputs:** All results/ outputs from notebooks 06-10.

**Outputs:** Final tables/figures under results/tables/ and results/figures/, ready to use.

**Runtime estimate:** 10-20 minutes.

## Apply consistent plot styling

In [1]:
import pandas as pd

from sdg_leakage_audit.utils.paths import results_figures_dir, results_tables_dir, to_project_relative
from sdg_leakage_audit.viz.style import apply_plot_style

apply_plot_style()

inflation_df = pd.read_csv(results_tables_dir() / "inflation_by_cell.csv")
sdg_impact_df = pd.read_csv(results_tables_dir() / "sdg_decision_impact.csv")
print(f"Loaded {len(inflation_df)} inflation rows and {len(sdg_impact_df)} SDG impact rows.")


Loaded 840 inflation rows and 35 SDG impact rows.


## Main results table (inflation per protocol/dataset/model)

In [2]:
main_results_table = (
    inflation_df[inflation_df["protocol_id"] != "P0"]
    .groupby(["protocol_id", "dataset_id"])["mae_inflation_pct"]
    .mean()
    .unstack("dataset_id")
    .round(2)
)

csv_path = results_tables_dir() / "main_results.csv"
tex_path = results_tables_dir() / "main_results.tex"
main_results_table.to_csv(csv_path)
main_results_table.to_latex(
    tex_path,
    caption="Mean MAE inflation (%) vs P0, by protocol and dataset.",
    label="tab:main-results",
    float_format="%.2f",
)
print(f"Saved -> {to_project_relative(csv_path)}")
print(f"Saved -> {to_project_relative(tex_path)}")
main_results_table


Saved -> results\tables\main_results.csv
Saved -> results\tables\main_results.tex


dataset_id,air_quality_uci,appliances_energy_prediction,beijing_multisite_air_quality,household_power_consumption,jena_climate
protocol_id,,,,,
P1,-28.02,25.27,2.05,12.29,-7.33
P2,0.09,-3.71,-2.38,-1.08,-0.07
P3,11.55,-3.56,0.88,-1.75,-0.51
P4,-1.34,3.47,4.46,7.75,-3.48
P5,5.86,3.28,2.66,-0.91,-2.64
P6,-21.39,27.86,7.23,11.24,-6.90


## Main results figure

In [3]:
from sdg_leakage_audit.viz.plots import plot_inflation_heatmap

plot_inflation_heatmap(
    main_results_table,
    title="Mean MAE inflation (%) vs P0, by protocol and dataset",
    cbar_label="MAE inflation (%)",
    save_path=results_figures_dir() / "11_main_inflation_heatmap",
)
print("Saved results/figures/11_main_inflation_heatmap.{png,pdf}")


2026-10-21 20:34:27 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\11_main_inflation_heatmap.png


2026-10-21 20:34:27 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\11_main_inflation_heatmap.pdf


Saved results/figures/11_main_inflation_heatmap.{png,pdf}


## SDG impact summary figure

In [4]:
sdg_impact_pivot = sdg_impact_df.pivot_table(
    index="dataset_id", columns="protocol_id", values="under_provisioning_pct"
).round(2)

plot_inflation_heatmap(
    sdg_impact_pivot,
    title="SDG decision-impact under-provisioning (%) vs P0, by dataset and protocol",
    cbar_label="under-provisioning (%)",
    save_path=results_figures_dir() / "11_sdg_impact_heatmap",
)
print("Saved results/figures/11_sdg_impact_heatmap.{png,pdf}")
sdg_impact_pivot


2026-10-21 20:34:28 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\11_sdg_impact_heatmap.png


2026-10-21 20:34:29 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\11_sdg_impact_heatmap.pdf


Saved results/figures/11_sdg_impact_heatmap.{png,pdf}


protocol_id,P0,P1,P2,P3,P4,P5,P6
dataset_id,,,,,,,
air_quality_uci,0.0,-16.42,3.86,16.47,5.47,7.46,-7.41
appliances_energy_prediction,0.0,25.33,-2.13,-1.93,2.98,1.17,26.22
beijing_multisite_air_quality,0.0,-8.47,-1.35,0.92,3.56,1.31,-5.47
household_power_consumption,0.0,9.58,-1.14,-1.37,7.22,-0.71,8.96
jena_climate,0.0,-33.65,0.48,-0.67,-0.41,-3.88,-33.81


## Export to final formats

In [5]:
exported_files = sorted(results_tables_dir().glob("main_results.*")) + sorted(
    results_figures_dir().glob("11_*")
)
print("Final files exported by this notebook:")
for f in exported_files:
    print(f"  {to_project_relative(f)}  ({f.stat().st_size:,} bytes)")

print(
    "\nNOTE: earlier notebooks (03, 05, 08, 10) saved figures as PNG only, not PNG+PDF -- "
    "this notebook's own new figures (11_*) follow the PNG+PDF, 300 dpi policy via "
    "viz.plots.save_figure_for_export, but retrofitting every earlier figure to also emit a "
    "PDF copy was out of scope for this pass. Flagged here rather than left unstated."
)


Final files exported by this notebook:
  results\tables\main_results.csv  (324 bytes)
  results\tables\main_results.tex  (622 bytes)
  results\figures\11_main_inflation_heatmap.pdf  (39,353 bytes)
  results\figures\11_main_inflation_heatmap.png  (227,579 bytes)
  results\figures\11_sdg_impact_heatmap.pdf  (39,868 bytes)
  results\figures\11_sdg_impact_heatmap.png  (188,517 bytes)

NOTE: earlier notebooks (03, 05, 08, 10) saved figures as PNG only, not PNG+PDF -- this notebook's own new figures (11_*) follow the PNG+PDF, 300 dpi policy via viz.plots.save_figure_for_export, but retrofitting every earlier figure to also emit a PDF copy was out of scope for this pass. Flagged here rather than left unstated.


## Findings / what to carry forward

- Main results table (mean MAE inflation % per protocol x dataset) exported as both CSV and
  LaTeX to `results/tables/main_results.{csv,tex}` -- ready to drop into the project's
  results write-up.
- Main results figure and the SDG decision-impact figure are both saved as PNG+PDF at 300 dpi
  (`results/figures/11_main_inflation_heatmap.*`, `results/figures/11_sdg_impact_heatmap.*`),
  via the new `viz.plots.save_figure_for_export` helper, matching the project's stated figure
  policy.
- Known gap, stated rather than hidden: earlier notebooks (03, 05, 08, 10) only ever saved PNG
  figures, not PNG+PDF -- the dual-format policy was only implemented starting with this
  notebook's own new figures. Retrofitting all earlier figures to also emit a PDF copy is
  straightforward (swap `_save_or_show` calls for `save_figure_for_export`) but was left as a
  follow-up rather than done opportunistically here.
- This completes the notebook sequence (00-11). Every numeric result produced across notebooks
  06-10 was generated under the `fast` experiment profile (single seed, 5% data subsample,
  minimal hyperparameter grids) -- every notebook's own findings cell has flagged this
  consistently, and it applies to this notebook's tables/figures too: they demonstrate the full
  pipeline and the inflation-measurement METHOD correctly, but a `full`-profile re-run (multiple
  seeds, complete hyperparameter grids, full data) is the necessary next step before any number
  produced by this repository is trustworthy, citable evidence rather than a pipeline-correctness result.
